# March Mania · Feature evidence, before another fit

**Milestone 01 — diagnose local changes, verify published results, inspect what reached the models.**

This is a **read-only investigation**, not another setup run or a model-training sweep. It leaves the repository, raw data, existing notebooks, branches and index unchanged. Small private copies preserve changed tracked-file versions; these copies are never included in the return ZIP.

Your uploaded milestone 00 has the expected commit and complete table-level checks, but `tracked_clean: false`. That does not tell us whether a source file changed or a notebook merely saved outputs. This notebook identifies the exact files rather than guessing.

**Historical score context:** submitted Brier **0.1222672**; research target **0.1097454**. Neither is a result of this notebook. Feature engineering remains open.

## 1 · Confirm the existing environment

Choose **Python (March Mania)**. Do not reinstall dependencies. Keep this folder beside the source repository, not inside it. Save and close project notebooks and shut down their other kernels before running this notebook.

In [ ]:
from pathlib import Path
import os, sys, json
import pandas as pd
import plotly.io as pio
from IPython.display import display, Markdown, FileLink

KIT = Path.cwd().resolve()
if not (KIT / "feature_evidence.py").is_file():
    KIT = Path.home() / "march_feature_audit"
REPO = Path(os.environ.get("MARCH_REPO", str(Path.home() / "march-machine-learning-mania-2026"))).expanduser().resolve()
PRIOR = Path(os.environ.get("MARCH_PRIOR_REPORTS", str(Path.home() / "march_workspace_sync/reports"))).expanduser().resolve()
REPORTS = KIT / "reports"
EXPECTED = "84b8fb36644a6558beded6dad84f5645ea4405d3"
MAX_SECONDS = 300  # Hard execution cap, not a runtime estimate.

assert (KIT / "feature_evidence.py").is_file(), "Open the notebook inside the extracted march_feature_audit folder."
assert (REPO / ".git").exists(), "Wrong repository location. Nothing should be deleted or cloned."
assert (PRIOR / "milestone_summary.json").is_file(), "Keep the original march_workspace_sync/reports folder."
assert Path(sys.prefix).resolve() == (REPO / ".venv").resolve(), "Select Python (March Mania), then restart this notebook. Do not reinstall."
sys.path.insert(0, str(KIT))
from feature_evidence import run_from_notebook
print("Kernel:", sys.executable)
print("Repository:", REPO)
print("Previous receipts:", PRIOR)
print("New reports:", REPORTS)
print("No project source modules will be imported or executed.")

## 2 · Run the bounded diagnostic once

The child process has a **300-second cap**, progress counters and **15-second heartbeats**. The notebook also has its own wall-clock watchdog. Verified report bytes are cached by SHA-256; reruns reuse those reports. No S3 archive or Kaggle download is needed.

Stages: tracked-file diagnosis → recheck the previously audited raw bytes → verify committed feature reports → save summary and Plotly figures.

**`REVIEW_REQUIRED` is an informative completed diagnosis, not a reason to reinstall or repeat the run.** The remaining cells are safe because they read committed historical evidence, never the changed project source. Training remains disabled in both status cases.

In [ ]:
summary = run_from_notebook(REPO, PRIOR, REPORTS, expected=EXPECTED, seconds=MAX_SECONDS)
print("Diagnostic status:", summary["status"])
print("Tracked changes:", summary["tracked_change_count"])
print("Data audit revalidated:", summary["data_preflight_revalidated"])
print("New models fitted:", summary["new_models_fitted"])
print("GitHub updated:", summary["github_updated"])
print("Repository state unchanged:", summary["repository_state_unchanged"])

## 3 · Identify exactly what changed

Notebook outputs, execution counts and execution timestamps are distinguished from source and metadata changes. Execution tags and kernel metadata are **not** silently dismissed. Staged and unstaged contents are inspected separately, including the case where the working file was changed back but a different version remains staged.

No file is reset, restored, stashed, deleted or committed. Review the classifications; do not assume an `environment_definition_change` or `code_or_config_change` is harmless.

In [ ]:
changes = pd.read_csv(REPORTS / "tracked_changes.csv")
if changes.empty:
    display(Markdown("**No tracked byte/mode changes found.** Hidden index flags or an in-progress Git operation, when present, still keep the gate closed."))
else:
    display(changes[["path", "classification", "staged", "unstaged"]])
    print("Private worktree/index backups were saved inside march_feature_audit/private_backups.")
    print("Keep those local. They are deliberately excluded from the return ZIP.")
checks = {k: summary[k] for k in ["head", "tracked_clean", "hidden_index_flag_count", "in_progress_git_operations", "repository_state_unchanged", "raw_tables_rechecked", "raw_data_unchanged", "historical_data_hashes_compared", "historical_data_hashes_matched", "computational_source_files_changed"]}
display(pd.DataFrame({"check": checks.keys(), "observed": [str(v) for v in checks.values()]}))

## 4 · Candidate catalog versus actual retained inputs

The public `feature_usage.csv` contains **retained full-block features from the completed feature-store study only**. It is not the complete rejection log and does not identify every input to the final submitted predictor.

The global catalog can contain inputs ineligible for women or absent in particular seasons. The second chart therefore reports **counts actually retained**, not a misleading percentage of a common eligible denominator. A zero means no observed retained inputs from that registered family in the corresponding fits; it is not proof that the family lacks signal.

In [ ]:
def show_chart(name):
    file = REPORTS / "figures" / (name + ".json")
    if file.is_file():
        fig = pio.from_json(file.read_text())
        fig.show()
    else:
        display(Markdown(f"No applicable recorded chart for `{name}`."))

show_chart("catalog")
show_chart("family_retention")
display(pd.read_csv(REPORTS / "usage_summary.csv"))
print("Catalog features:", summary["catalog_features"])
print("Full-bank fits:", summary["full_block_fits"])
print("Retained per full-bank fit:", summary["retained_per_full_fit_min"], "to", summary["retained_per_full_fit_max"])
print("Distinct retained at least once across these fits:", summary["distinct_retained_full_block"])

## 5 · Inspect one population/model at a time

Start with men/logistic. To inspect another route, change `ROUTE` to `M/hist`, `W/logistic`, or `W/hist` and rerun this and the following plotting cells. **This only redraws saved evidence; it never fits anything.** The standalone HTML includes all recorded routes.

Retention reflects the selection pipeline. It is not causal importance. The top-30 stability chart is descriptive and must not be used as a globally selected input list for later out-of-fold evaluation.

In [ ]:
ROUTE = "M/logistic"
valid_routes = set(pd.read_csv(REPORTS / "usage_summary.csv")["route"])
assert ROUTE in valid_routes, f"Choose one of {sorted(valid_routes)}"
prefix = ROUTE.replace("/", "_")
show_chart(prefix + "_screening")
show_chart(prefix + "_stability")

## 6 · Historical Brier: keep populations, folds and weighting separate

These curves show **recorded historical fixed-recipe studies**, not a new evaluation, not the final submitted model, and not the 2026 leaderboard. Mean-season Brier weights each season equally; game-weighted Brier weights it by its recorded number of games. The two aggregates are saved separately.

Already-inspected seasons are exploratory evidence. A small minimum over many tested blocks is not reliable proof of future improvement.

In [ ]:
show_chart(prefix + "_brier")
metrics = pd.read_csv(REPORTS / "metric_summary.csv")
gender, model = ROUTE.split("/")
anchor_blocks = ["strength", "baseline_124", "full", "rankings", "conference"]
view = metrics[(metrics.Gender == gender) & (metrics.model == model) & metrics.block.isin(anchor_blocks)]
display(view[["block", "mean_season_brier", "game_weighted_brier", "seasons", "games"]].sort_values("block"))

## 7 · What the existing ablations can and cannot establish

**Delta = Brier(without family) − Brier(full).** Positive values favor the full pipeline; negative values favor the family-removed pipeline.

Existing omission runs also reran selection and could fill freed positions with other features. These are **conditional pipeline effects**, not isolated effects of removing only that family. The code only compares recorded season rows with equal game counts. Without prediction-row IDs, even equal counts are not proof of exact paired row identity.

The next scientific comparison should freeze the compact anchor, preprocessing and model settings, then test one small, domain-defined addition or forced-inclusion block using only permitted training information. No automated feature retention decision is made here.

In [ ]:
show_chart(prefix + "_ablations")
ablations = pd.read_csv(REPORTS / "ablations.csv")
view = ablations[(ablations.Gender == gender) & (ablations.model == model)]
display(view[["Season", "omitted_family", "brier", "full_brier", "delta_without_minus_full", "comparable_counts"]].sort_values(["omitted_family", "Season"]))

## 8 · Save, return the evidence, and stop

**One return file:** `reports/milestone_01_return.zip`. It includes the exact changed-file classifications, feature retention summaries, historical aggregate metrics and report provenance. It excludes private backups, credentials, raw rows and fitted models.

`REVIEW_REQUIRED`: the diagnostic completed but local changes or data mismatches remain. Do not run a reset, stash or install to hide the issue. Return the ZIP for a targeted next instruction.

`PASS_DIAGNOSTIC`: source/data checks passed and historical evidence was summarized. This is **not permission to start a full training sweep**. Feature family value and final-model lineage still need controlled investigation.

When a preceding cell fails with `STOP`, attach `reports/failure.json` instead; do not keep retrying unchanged.

In [ ]:
print("Status:", summary["status"])
print("Historical submitted Brier (unchanged):", summary["current_submitted_brier"])
print("New models fitted:", summary["new_models_fitted"])
print("GitHub updated:", summary["github_updated"])
print("Next:", summary["next_step"])
print("Save this executed notebook with Ctrl+S.")
for filename in ["milestone_01_return.zip", "milestone_01_summary.json", "feature_evidence.html"]:
    display(FileLink(str(Path("reports") / filename)))
print("Stop here. Do not run canonical notebook 02 in training mode.")